# 1.3.2. TensorFlow Decision Forests banko klientams

**Liutauras Turauskis · DIPf-24 · ND1 tęsinys**

Pritaikyta pagal dėstytojo pateiktą `AdventureWorksDW_Decision_Forests.ipynb`. Pavyzdžio eiga paaiškinta prie atitinkamų žingsnių;
programos komentarai paaiškina svarbiausias operacijas. Išlaikytas tikras TensorFlow Decision Forests modelis, medžio vaizdavimas, OOB analizė, vertinimas ir išsaugojimas.

**Paleidimas:** Google Colab → „Runtime“ → „Run all“. Ląsteles vykdyti nuo pradžios.
Duomenys atsisiunčiami automatiškai; savo kodo, failų ir API rakto įterpti nereikia.
GPU rekomenduojamas vaizdų mokymui, bet nėra būtinas. Pirmas atsisiuntimas gali užtrukti.

**Rezultatų sąžiningumas:** šioje pateikiamoje kopijoje mokymo išvestys išvalytos.
Rodikliai, grafikai ir konkrečios išvados apskaičiuojami paleidus, o ne įrašomi iš anksto.
Parengimo aplinkoje TensorFlow mokymas nebuvo vykdytas; prieš pateikiant reikia paleisti visą užrašyną
Colab ir išsaugoti kopiją su išvestimis. Atsitiktinė sėkla mažina rezultatų svyravimą, tačiau GPU
ir bibliotekų versijos gali lemti nedidelius skirtumus.


In [ ]:
%pip -q install "pandas>=2.2,<3" "scikit-learn>=1.5,<2" "matplotlib>=3.8,<4" requests pillow


In [ ]:
from pathlib import Path
import io, json, zipfile, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix, ConfusionMatrixDisplay,
    classification_report)
from IPython.display import display
SEED = 42
OUT = Path('rezultatai')
OUT.mkdir(exist_ok=True)


## 1. Kiti verslo duomenys ir prognozės tikslas

Originale buvo „AdventureWorks“ klientai ir `BikeBuyer` (ar pirko dviratį).
Čia naudojami **UCI Bank Marketing** duomenys: ar klientas užsisakė terminuotąjį indėlį (`y`).
Tai viešas pasirinktas, o ne mano paties surinktas rinkinys. Originalūs autoriai:
S. Moro, P. Rita, P. Cortez; [UCI šaltinis ir CC BY 4.0 licencija](https://archive.ics.uci.edu/dataset/222/bank+marketing).
Verslo rinkinys taip pat platinamas [Kaggle](https://www.kaggle.com/datasets/janiobachmann/bank-marketing-dataset);
čia naudojamas UCI originalas, kad nereikėtų Kaggle prisijungimo.

Pasirenkama iki 8 000 įrašų iš `bank-full.csv`, išlaikant klasių proporcijas.
`duration` pašalinamas: skambučio trukmė nežinoma prieš skambutį ir lemtų informacijos nutekėjimą.
Likę kontaktų bei ankstesnės kampanijos požymiai naudojami kaip turima kliento informacija.
Tai mokomasis atsitiktinio padalijimo eksperimentas; būsimos kampanijos kokybei reikėtų laikinio padalijimo.


In [ ]:
URL = 'https://archive.ics.uci.edu/static/public/222/bank+marketing.zip'
r = requests.get(URL, timeout=90)
r.raise_for_status()
# UCI archyve bank-full.csv gali būti papildomame bank.zip archyve.
def read_bank_csv(blob):
    with zipfile.ZipFile(io.BytesIO(blob)) as archive:
        names = archive.namelist()
        csv_name = next((n for n in names if n.endswith('bank-full.csv')), None)
        if csv_name:
            return pd.read_csv(archive.open(csv_name), sep=';')
        nested = next((n for n in names if n.rsplit('/', 1)[-1] == 'bank.zip'), None)
        if nested:
            return read_bank_csv(archive.read(nested))
    raise ValueError('UCI archyve nerastas bank-full.csv; patikrinkite šaltinį.')

raw = read_bank_csv(r.content)
assert set(raw['y']) == {'yes', 'no'}
df = raw.drop(columns=['duration']).copy()
df['target'] = df.pop('y').map({'no': 0, 'yes': 1}).astype('int32')
if len(df) > 8000:
    df, _ = train_test_split(df, train_size=8000, stratify=df['target'], random_state=SEED)
train, remaining = train_test_split(df, test_size=0.30, stratify=df['target'], random_state=SEED)
val, test = train_test_split(remaining, test_size=0.50, stratify=remaining['target'], random_state=SEED)
assert set(train.index).isdisjoint(val.index) and set(train.index).isdisjoint(test.index)
assert set(val.index).isdisjoint(test.index)
assert 'target' not in train.drop(columns='target') and 'duration' not in df
display(df.head())
display(pd.DataFrame({k: {'įrašai': len(v), 'indėlis_1_dalis': v.target.mean()}
                      for k, v in {'mokymas': train, 'validavimas': val, 'testas': test}.items()}).T)
display(df.dtypes.rename('tipas').to_frame())
print('Trūkstamos reikšmės:', int(df.isna().sum().sum()))
# unknown paliekamas kaip atskira kategorija; pdays=-1 reiškia, kad anksčiau nekontaktuota.
assert not df.isna().any().any()
for name, part in [('train', train), ('val', val), ('test', test)]:
    part.to_csv(OUT / f'bank_{name}.csv', index=False)


## 2. Suderinama TFDF aplinka

TFDF turi TensorFlow versijai jautrius C++ plėtinius. [Oficialus 1.12.0 leidimas](https://github.com/tensorflow/decision-forests/releases/tag/v1.12.0)
skirtas TensorFlow 2.19.0 ir palaiko Python 3.12. Todėl Colab Linux aplinkoje sukuriama izoliuota
Python 3.12 aplinka; pagrindinio Colab TensorFlow perrašyti ir sesijos perkrauti nereikia.
Pirmas įdiegimas atsisiunčia nemažai priklausomybių. Tai įprastas TFDF mokymas CPU, GPU jam nereikalingas.
Originale naudotas `dtreeviz` blokas buvo nepilnai sujungtas; čia medį rodo paties TFDF HTML.


In [ ]:
%pip -q install uv


In [ ]:
import sys, subprocess, platform
assert platform.system() == 'Linux', 'Šį TFDF užrašyną paleiskite Google Colab (Linux).'
ENV = Path('/content/nd1_tfdf_py312')
PY = ENV / 'bin/python'
if not PY.exists():
    subprocess.run([sys.executable, '-m', 'uv', 'venv', '--python', '3.12', str(ENV)], check=True)
subprocess.run([sys.executable, '-m', 'uv', 'pip', 'install', '--python', str(PY),
    'tensorflow==2.19.0', 'tf-keras==2.19.0', 'tensorflow-decision-forests==1.12.0',
    'ydf==0.11.0', 'protobuf>=5.29.1,<6',
    'pandas>=2.2,<3', 'scikit-learn>=1.5,<2', 'numpy>=1.26,<2.2'], check=True)


## 3. Miško mokymas, tikrinimas ir interpretavimas

Originalo dešimties medžių miškas pakeistas į 100 medžių, iki 12 lygių gylio.
Miškas pats apdoroja kategorinius požymius, todėl one-hot ir normalizavimas nereikalingi.
`fit` augina medžius vienu procesu; tai nėra epochomis mokomas neuroninis tinklas.

Žemiau esantis skriptas aiškiai atskiria mokymą, slenksčio pasirinkimą validavime ir testą.
Jis taip pat išsaugo OOB (mokymo įrašų, nepatekusių į konkrečių medžių imtis) statistiką,
požymių svarbą ir medį. OOB vertinimas nepakeičia galutinio nepriklausomo testo.


In [ ]:
TRAIN_SCRIPT = """
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'
import json
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
import tf_keras
import tensorflow_decision_forests as tfdf
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score
out = Path('rezultatai')
train, val, test = [pd.read_csv(out / f'bank_{s}.csv') for s in ['train', 'val', 'test']]
train_ds = tfdf.keras.pd_dataframe_to_tf_dataset(train, label='target')
forest = tfdf.keras.RandomForestModel(num_trees=100, max_depth=12,
    min_examples=5, random_seed=42, verbose=1)
forest.fit(train_ds)
forest.compile(metrics=['accuracy'])
evaluation = forest.evaluate(tfdf.keras.pd_dataframe_to_tf_dataset(test, label='target'), return_dict=True)
print('TFDF testas:', evaluation)
# Prognozės duomenų rinkinyje nėra target stulpelio.
def ds(frame):
    return tfdf.keras.pd_dataframe_to_tf_dataset(frame.drop(columns='target'))
pv = forest.predict(ds(val), verbose=0).ravel()
pt = forest.predict(ds(test), verbose=0).ravel()
thresholds = np.linspace(0.05, 0.95, 91)
scores = [f1_score(val.target, pv >= t, zero_division=0) for t in thresholds]
threshold = float(thresholds[int(np.argmax(scores))])
pred = pt >= threshold
metrics = dict(threshold=threshold, accuracy=accuracy_score(test.target, pred),
    precision=precision_score(test.target, pred, zero_division=0),
    recall=recall_score(test.target, pred, zero_division=0), f1=f1_score(test.target, pred),
    roc_auc=roc_auc_score(test.target, pt), pr_auc_AP=average_precision_score(test.target, pt))
(out / 'df_rodikliai.json').write_text(json.dumps(metrics, indent=2))
pd.DataFrame({'target': test.target, 'probability': pt, 'prediction': pred.astype(int)}).to_csv(out / 'df_prognozes.csv', index=False)
inspector = forest.make_inspector()
logs = inspector.training_logs()
pd.DataFrame([{'trees': x.num_trees, 'oob_accuracy': x.evaluation.accuracy,
               'oob_loss': x.evaluation.loss} for x in logs]).to_csv(out / 'df_oob.csv', index=False)
# Vienas medis paaiškina vieną miško dalį, o ne visą kolektyvinę prognozę.
html = tfdf.model_plotter.plot_model(forest, tree_idx=0, max_depth=3)
(out / 'df_medis.html').write_text(html, encoding='utf-8')
importance = inspector.variable_importances()
key = 'NUM_AS_ROOT' if 'NUM_AS_ROOT' in importance else next(iter(importance))
pd.DataFrame([{'feature': f.name, 'importance': value} for f, value in importance[key]]).to_csv(out / 'df_svarba.csv', index=False)
(out / 'df_svarbos_tipas.txt').write_text(key)
# TFDF naudoja su TF 2.19 suderinamą tf_keras, ne Keras 3 .keras formatą.
forest.save(str(out / 'bank_forest'), overwrite=True)
restored = tf_keras.models.load_model(str(out / 'bank_forest'))
np.testing.assert_allclose(pt, restored.predict(ds(test), verbose=0).ravel(), atol=1e-6)
print('Išsaugoto ir įkelto miško prognozės sutapo.')
print(json.dumps(metrics, indent=2))
"""
SCRIPT = OUT / "mokyti_tfdf.py"
SCRIPT.write_text(TRAIN_SCRIPT, encoding="utf-8")
subprocess.run([str(PY), str(SCRIPT)], check=True)


## 4. Rezultatai ir grafikai

Tikslumas lyginamas su daugumos klasės etalonu. Klasių disbalansui vertinti pateikiami ir F1 bei AP.
Požymių svarba apibūdina modelio elgesį; ji neįrodo priežastinio poveikio indėlio pasirinkimui.
Interaktyviame medyje sąlygos viršuje skaido klientus į grupes. Lapai turi klasių pasiskirstymus.


In [ ]:
from IPython.display import HTML
metrics = json.loads((OUT / 'df_rodikliai.json').read_text())
display(pd.DataFrame([metrics], index=['TFDF']))
print('Daugumos klasės etaloninis accuracy:', (test.target == int(train.target.mode().iloc[0])).mean())
predictions = pd.read_csv(OUT / 'df_prognozes.csv')
ConfusionMatrixDisplay.from_predictions(predictions.target, predictions.prediction,
    display_labels=['ne', 'taip'], cmap='Greens')
plt.title('TFDF nepriklausomas testas'); plt.show()
logs = pd.read_csv(OUT / 'df_oob.csv')
fig, axes = plt.subplots(1, 2, figsize=(11, 3))
for ax, metric in zip(axes, ['oob_accuracy', 'oob_loss']):
    ax.plot(logs.trees, logs[metric]); ax.set(xlabel='Medžių skaičius', title=metric)
plt.tight_layout(); plt.show()
importance = pd.read_csv(OUT / 'df_svarba.csv').sort_values('importance')
importance.plot.barh(x='feature', y='importance', legend=False, figsize=(7, 5))
plt.title('Požymių svarba: ' + (OUT / 'df_svarbos_tipas.txt').read_text()); plt.tight_layout(); plt.show()
display(HTML((OUT / 'df_medis.html').read_text(encoding='utf-8')))
nn_file = OUT / 'nn_rodikliai.json'
if nn_file.exists():
    display(pd.DataFrame([json.loads(nn_file.read_text()), metrics], index=['NN', 'TFDF']))
else:
    print('NN rodiklių šioje sesijoje nėra; jie pateikiami 1.3.1 užrašyne.')
print(f'TFDF testo F1={metrics["f1"]:.3f}, ROC AUC={metrics["roc_auc"]:.3f}.')


## 5. Ką parodo palyginimas?

NN mokosi skaitinių svorių, o miškas sujungia daugelio sprendimų medžių išvadas.
Abiejuose 1.3 failuose naudojamos tos pačios 8 000 eilučių parinkimo ir 70/15/15 skaidymo taisyklės.
Sąžiningai lyginti reikia abiejų užrašynų tikrus testo rodiklius, o ne vien mokymo accuracy.
Nėra garantijos, kad sudėtingesnis algoritmas bus geresnis. OOB kreivės stabilizavimasis rodo,
ar papildomi medžiai dar keičia vidinį įvertį.

Šaltiniai: [TFDF pradžiamokslis](https://www.tensorflow.org/decision_forests/tutorials/beginner_colab),
[diegimas ir suderinamumas](https://www.tensorflow.org/decision_forests/installation).
